In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# show all columns nicely
pd.set_option("display.max_columns", None)
print("Libraries imported successfully.")

Libraries imported successfully.


### 1. Create the raw dataset

df = pd.read_csv("students.csv")

In [3]:
data = {
    "Name":       ["Asha", "Ravi", "Meena", "John", "Sara", "Karan",
                   "Asha", "Priya", "Aman", "Nina", "Vikram", "Tara"],
    "Gender":     ["F", "M", "F", "M", "F", "M",
                   "F", "F", "M", "F", "M", "F"],
    "City":       ["Pune", "Delhi", "Pune", "Mumbai", "Delhi", "Mumbai",
                   "Pune", "Delhi", "Pune", "Mumbai", "Delhi", "Pune"],
    "Age":        [20, 21, 22, 20, 23, 21, 20, 22, np.nan, 21, 24, 20],
    "StudyHours": [5, 3, 6, 2, 7, np.nan, 5, 6, 1, 4, 8, 5],
    "Attendance": [85, 70, 90, 60, 95, 65, 85, 88, 55, 75, 92, 80],
    "Marks":      [78, 55, 88, 40, 92, np.nan, 78, 85, 35, 60, 95, 75],
    "Result":     [1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 1],
}

df = pd.DataFrame(data)
df

,Name,Gender,City,Age,StudyHours,Attendance,Marks,Result
0,Asha,F,Pune,20.0,5.0,85,78.0,1
1,Ravi,M,Delhi,21.0,3.0,70,55.0,0
2,Meena,F,Pune,22.0,6.0,90,88.0,1
3,John,M,Mumbai,20.0,2.0,60,40.0,0
4,Sara,F,Delhi,23.0,7.0,95,92.0,1
5,Karan,M,Mumbai,21.0,NaN,65,NaN,0
6,Asha,F,Pune,20.0,5.0,85,78.0,1
7,Priya,F,Delhi,22.0,6.0,88,85.0,1
8,Aman,M,Pune,NaN,1.0,55,35.0,0
9,Nina,F,Mumbai,21.0,4.0,75,60.0,0


In [4]:
print("Shape (rows, columns):", df.shape)

Shape (rows, columns): (12, 8)


In [5]:
df.head()

,Name,Gender,City,Age,StudyHours,Attendance,Marks,Result
0,Asha,F,Pune,20.0,5.0,85,78.0,1
1,Ravi,M,Delhi,21.0,3.0,70,55.0,0
2,Meena,F,Pune,22.0,6.0,90,88.0,1
3,John,M,Mumbai,20.0,2.0,60,40.0,0
4,Sara,F,Delhi,23.0,7.0,95,92.0,1


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Name        12 non-null     object 
 1   Gender      12 non-null     object 
 2   City        12 non-null     object 
 3   Age         11 non-null     float64
 4   StudyHours  11 non-null     float64
 5   Attendance  12 non-null     int64  
 6   Marks       11 non-null     float64
 7   Result      12 non-null     int64  
dtypes: float64(3), int64(2), object(3)
memory usage: 900.0+ bytes


In [7]:
print("Missing values in each column:")
print(df.isnull().sum())

Missing values in each column:
Name          0
Gender        0
City          0
Age           1
StudyHours    1
Attendance    0
Marks         1
Result        0
dtype: int64


In [8]:
print("Number of duplicate rows:", df.duplicated().sum())

Number of duplicate rows: 1


In [9]:
df = df.drop(columns=["Name"])
print("Columns now:", list(df.columns))
print("Shape:", df.shape)

Columns now: ['Gender', 'City', 'Age', 'StudyHours', 'Attendance', 'Marks', 'Result']
Shape: (12, 7)


In [10]:
print("Shape BEFORE removing duplicates:", df.shape)

df = df.drop_duplicates().reset_index(drop=True)

print("Shape AFTER  removing duplicates:", df.shape)

Shape BEFORE removing duplicates: (12, 7)
Shape AFTER  removing duplicates: (11, 7)


In [11]:
number_cols = ["Age", "StudyHours", "Attendance", "Marks"]

print("Missing values BEFORE filling:")
print(df[number_cols].isnull().sum(), "\n")

# Fill each numeric column's blanks with that column's mean
for col in number_cols:
    df[col] = df[col].fillna(df[col].mean())

print("Missing values AFTER filling:")
print(df[number_cols].isnull().sum())

Missing values BEFORE filling:
Age           1
StudyHours    1
Attendance    0
Marks         1
dtype: int64 

Missing values AFTER filling:
Age           0
StudyHours    0
Attendance    0
Marks         0
dtype: int64


In [12]:
# Confirm there are no missing values left anywhere
print("Total missing values in the whole table:", df.isnull().sum().sum())
df

Total missing values in the whole table: 0


,Gender,City,Age,StudyHours,Attendance,Marks,Result
0,F,Pune,20.0,5.0,85,78.0,1
1,M,Delhi,21.0,3.0,70,55.0,0
2,F,Pune,22.0,6.0,90,88.0,1
3,M,Mumbai,20.0,2.0,60,40.0,0
4,F,Delhi,23.0,7.0,95,92.0,1
5,M,Mumbai,21.0,4.7,65,70.3,0
6,F,Delhi,22.0,6.0,88,85.0,1
7,M,Pune,21.4,1.0,55,35.0,0
8,F,Mumbai,21.0,4.0,75,60.0,0
9,M,Delhi,24.0,8.0,92,95.0,1


In [13]:
print("Shape BEFORE encoding:", df.shape)

df = pd.get_dummies(df, columns=["Gender", "City"], drop_first=True)

print("Shape AFTER  encoding:", df.shape)
print("Columns now:", list(df.columns))
df.head()

Shape BEFORE encoding: (11, 7)
Shape AFTER  encoding: (11, 8)
Columns now: ['Age', 'StudyHours', 'Attendance', 'Marks', 'Result', 'Gender_M', 'City_Mumbai', 'City_Pune']


,Age,StudyHours,Attendance,Marks,Result,Gender_M,City_Mumbai,City_Pune
0,20.0,5.0,85,78.0,1,False,False,True
1,21.0,3.0,70,55.0,0,True,False,False
2,22.0,6.0,90,88.0,1,False,False,True
3,20.0,2.0,60,40.0,0,True,True,False
4,23.0,7.0,95,92.0,1,False,False,False


In [14]:
X = df.drop(columns=["Result"])   # inputs
y = df["Result"]                  # answer

print("Feature matrix X shape:", X.shape)
print("Target y shape:        ", y.shape)
print("\nFeature columns:", list(X.columns))

Feature matrix X shape: (11, 7)
Target y shape:         (11,)

Feature columns: ['Age', 'StudyHours', 'Attendance', 'Marks', 'Gender_M', 'City_Mumbai', 'City_Pune']


In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,      # 25% goes to the test set
    random_state=42      # fixed number so everyone gets the same split
)

print("Training set:", X_train.shape)
print("Testing  set:", X_test.shape)

Training set: (8, 7)
Testing  set: (3, 7)


In [16]:
scaler = StandardScaler()

# LEARN the mean/std from training data and apply it
X_train_scaled = scaler.fit_transform(X_train)

# APPLY the same training mean/std to the test data (no new learning)
X_test_scaled = scaler.transform(X_test)

print("Scaled training matrix shape:", X_train_scaled.shape)
print("Scaled testing  matrix shape:", X_test_scaled.shape)

Scaled training matrix shape: (8, 7)
Scaled testing  matrix shape: (3, 7)


In [17]:
X_train_clean = pd.DataFrame(X_train_scaled, columns=X.columns)
print("CLEAN FEATURE MATRIX (training data, after scaling):")
X_train_clean.round(2)

CLEAN FEATURE MATRIX (training data, after scaling):


,Age,StudyHours,Attendance,Marks,Gender_M,City_Mumbai,City_Pune
0,-1.35,0.38,0.25,0.42,-0.77,-0.58,1.29
1,0.73,0.88,0.99,1.06,-0.77,-0.58,1.29
2,-0.31,-0.63,-0.49,-0.55,1.29,-0.58,-0.77
3,-0.31,-0.13,-0.12,-0.30,-0.77,1.73,-0.77
4,1.76,1.39,1.37,1.25,-0.77,-0.58,-0.77
5,0.10,-1.64,-1.61,-1.52,1.29,-0.58,1.29
6,-1.35,-1.13,-1.24,-1.27,1.29,1.73,-0.77
7,0.73,0.88,0.85,0.91,-0.77,-0.58,-0.77


In [18]:
# Proof it worked: each column's mean is ~0 and std is ~1
print("Column means after scaling (should be ~0):")
print(X_train_clean.mean().round(2))
print("\nColumn std after scaling (should be ~1):")
print(X_train_clean.std(ddof=0).round(2))

Column means after scaling (should be ~0):
Age           -0.0
StudyHours     0.0
Attendance    -0.0
Marks          0.0
Gender_M      -0.0
City_Mumbai   -0.0
City_Pune     -0.0
dtype: float64

Column std after scaling (should be ~1):
Age            1.0
StudyHours     1.0
Attendance     1.0
Marks          1.0
Gender_M       1.0
City_Mumbai    1.0
City_Pune      1.0
dtype: float64


In [19]:
# Rebuild the raw, messy table from scratch
raw = pd.DataFrame(data).drop(columns=["Name"]).drop_duplicates().reset_index(drop=True)

X_raw = raw.drop(columns=["Result"])
y_raw = raw["Result"]

num_features = ["Age", "StudyHours", "Attendance", "Marks"]
cat_features = ["Gender", "City"]

# Steps for the number columns: fill blanks, then scale
numeric_steps = Pipeline([
    ("fill_missing", SimpleImputer(strategy="mean")),
    ("scale",        StandardScaler())
])

# Steps for the text columns: one-hot encode
categorical_steps = Pipeline([
    ("encode", OneHotEncoder(drop="first", handle_unknown="ignore"))
])

# Apply the right steps to the right columns
preprocessor = ColumnTransformer([
    ("numbers", numeric_steps,     num_features),
    ("text",    categorical_steps, cat_features)
])

print("Preprocessor built.")

Preprocessor built.


In [20]:
# Split first (no leakage), then let the pipeline do all the cleaning
Xtr, Xte, ytr, yte = train_test_split(
    X_raw, y_raw, test_size=0.25, random_state=42
)

# fit_transform on train (learn + apply), transform on test (apply only)
Xtr_ready = preprocessor.fit_transform(Xtr)
Xte_ready = preprocessor.transform(Xte)

print("Training matrix after full pipeline:", Xtr_ready.shape)
print("Testing  matrix after full pipeline:", Xte_ready.shape)
print("\nOne object handled missing values + encoding + scaling, in order.")

Training matrix after full pipeline: (8, 7)
Testing  matrix after full pipeline: (3, 7)

One object handled missing values + encoding + scaling, in order.
